In [ ]:
import os, sys, json, subprocess, time, io, gc, shutil, glob
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import requests as http_requests

NOTEBOOK_NAME = "watermark-remover-pt-5"
STEP_NAME = "step_watermark_pt5"

print("Instalando dependencias...")
os.system("apt-get install -y ffmpeg > /dev/null 2>&1")

def _load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try: return _s.get_secret(name)
            except: return ""
        print("Carregando Kaggle Secrets...")
        return _get
    except ImportError:
        from dotenv import load_dotenv; load_dotenv()
        return lambda name: os.getenv(name, "")

_get = _load_secrets()
DRIVE_REFRESH_TOKEN = _get("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID = _get("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET = _get("DRIVE_CLIENT_SECRET")
HF_TOKEN = _get("HF_TOKEN")
DATABASE_URL = _get("DATABASE_URL")
PROJECT_ID = _get("PIPELINE_PROJECT_ID")
PIPELINE_WEBHOOK_URL = _get("PIPELINE_WEBHOOK_URL")

print("Autenticando Drive...")
try:
    _creds = Credentials(token=None, refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID, client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"])
    _creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=_creds)
    print("Drive autenticado!")
except Exception as e:
    drive_service = None
    print(f"Falha Drive: {e}")

def _buscar_id(caminho):
    partes = caminho.strip("/").split("/")
    pid = "root"
    for p in partes:
        q = f"name='{p}' and '{pid}' in parents and trashed=false"
        r = drive_service.files().list(q=q, fields="files(id,mimeType)", orderBy="modifiedTime desc").execute()
        a = r.get("files", [])
        if not a: return None
        pid = a[0]["id"]
    return pid

def _garantir_pasta(caminho):
    partes = caminho.strip("/").split("/")
    pid = "root"
    for p in partes:
        q = f"name='{p}' and '{pid}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
        r = drive_service.files().list(q=q, fields="files(id)").execute()
        e = r.get("files", [])
        if e: pid = e[0]["id"]
        else:
            nova = drive_service.files().create(body={"name": p, "mimeType": "application/vnd.google-apps.folder", "parents": [pid]}, fields="id").execute()
            pid = nova["id"]
    return pid

def baixar_do_drive(caminho_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        fid = _buscar_id(caminho_drive)
        if not fid: print(f"  Nao encontrado: {caminho_drive}"); return False
        req = drive_service.files().get_media(fileId=fid)
        os.makedirs(os.path.dirname(destino_local) or ".", exist_ok=True)
        with open(destino_local, "wb") as fh:
            dl = MediaIoBaseDownload(fh, req); done = False
            while not done: _, done = dl.next_chunk()
        print(f"  Baixado: {caminho_drive}"); return True
    except Exception as ex: print(f"  Erro: {caminho_drive}: {ex}"); return False

def salvar_no_drive(caminho_local, caminho_drive):
    if not drive_service or not os.path.exists(caminho_local): return
    try:
        partes = caminho_drive.strip("/").split("/")
        nome = partes[-1]; pasta = "/".join(partes[:-1]) if len(partes) > 1 else ""
        pid = _garantir_pasta(pasta) if pasta else "root"
        q = f"name='{nome}' and '{pid}' in parents and trashed=false"
        r = drive_service.files().list(q=q, fields="files(id)").execute()
        e = r.get("files", []); media = MediaFileUpload(caminho_local, resumable=True)
        if e: drive_service.files().update(fileId=e[0]["id"], media_body=media).execute()
        else: drive_service.files().create(body={"name": nome, "parents": [pid]}, media_body=media, fields="id").execute()
        print(f"  Salvo: {caminho_drive}")
    except Exception as ex: print(f"  Erro salvar {caminho_drive}: {ex}")

_cell_timers = {}
def _db_exec(query, params):
    if not DATABASE_URL: return
    try:
        import psycopg2; conn = psycopg2.connect(DATABASE_URL); cur = conn.cursor()
        cur.execute(query, params); conn.commit(); cur.close(); conn.close()
    except: pass

def cell_start(idx, name=""):
    _cell_timers[idx] = time.time()
    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")
    if not PROJECT_ID: return
    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,'running',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))
    _db_exec("UPDATE pipeline_cell_tracking SET status='running',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))

def cell_end(idx, status="done", msg=""):
    elapsed = ""
    if idx in _cell_timers:
        s = int(time.time() - _cell_timers.pop(idx))
        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"
    icon = "OK" if status == "done" else "ERRO"
    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'─'*50}")
    if not PROJECT_ID: return
    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))

def report_step(status, msg=""):
    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={"project_id": PROJECT_ID, "step": STEP_NAME, "status": status, "message": msg}, timeout=15)
        except: pass
    if not PROJECT_ID: return
    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))
    _db_exec("INSERT INTO pipeline_logs (project_id,step,status,message) VALUES (%s::uuid,%s,%s,%s)", (PROJECT_ID, STEP_NAME, status, msg))

DRIVE_ATIVO = "DRAMA/PIPELINE/ATIVO"
DRIVE_WATERMARK = "DRAMA/PIPELINE/WATERMARK"
DRIVE_ENHANCER = "DRAMA/PIPELINE/ENHANCER"
DRIVE_OMNI = "DRAMA/PIPELINE/OMNI"
DRIVE_RENDER = "DRAMA/PIPELINE/RENDER"
DRIVE_FINAL = "DRAMA/PIPELINE/FINAL"
BASE_PATH = "/kaggle/working"
os.makedirs(BASE_PATH, exist_ok=True)

print("Baixando pacote de fontes do Google Drive...")
os.system("mkdir -p /usr/share/fonts/truetype/custom")
try:
    _fid = _buscar_id("DRAMA/PIPELINE/FONTS")
    if _fid:
        _r = drive_service.files().list(q=f"'{_fid}' in parents and trashed=false", fields="files(id, name)").execute()
        _f_list = _r.get("files", [])
        print(f"  {len(_f_list)} fontes encontradas no Drive")
        for _f in _f_list:
            _dest = f"/usr/share/fonts/truetype/custom/{_f['name']}"
            if os.path.exists(_dest):
                print(f"  Ja existe: {_f['name']}")
                continue
            try:
                _req = drive_service.files().get_media(fileId=_f['id'])
                with open(_dest, "wb") as _fh:
                    _dl = MediaIoBaseDownload(_fh, _req); _done = False
                    while not _done: _, _done = _dl.next_chunk()
                print(f"  Baixado: {_f['name']}")
            except Exception as _ex:
                print(f"  Erro baixando {_f['name']}: {_ex}")
    else:
        print("  ⚠️ Pasta DRAMA/PIPELINE/FONTS nao encontrada no Drive!")
except Exception as e:
    print(f"  ❌ Erro ao baixar fontes do Drive: {e}")
os.system("fc-cache -f -v > /dev/null 2>&1")
_custom_dir = "/usr/share/fonts/truetype/custom"
if os.path.isdir(_custom_dir):
    _installed = os.listdir(_custom_dir)
    print(f"  Fontes instaladas ({len(_installed)}): {_installed}")
else:
    print("  ⚠️ Diretorio de fontes custom nao existe!")
print("Setup de fontes concluido!")

cell_end(0, "done", "Setup concluido")

In [ ]:
cell_start(1, 'Download dos Arquivos')

import cv2, numpy as np
baixar_do_drive(f"{DRIVE_ATIVO}/video_pt5.mp4", f"{BASE_PATH}/video_pt5.mp4")
baixar_do_drive(f"{DRIVE_ATIVO}/mask.png", f"{BASE_PATH}/mask.png")
print("Arquivos prontos!")

cell_end(1, 'done', 'Download dos Arquivos concluido')

In [ ]:
cell_start(2, 'Processamento Watermark')

INPUT = f"{BASE_PATH}/video_pt5.mp4"
MASK = f"{BASE_PATH}/mask.png"
OUTPUT = f"{BASE_PATH}/pt5_limpo.mp4"

import cv2, numpy as np

# Se mask não existe, copiar direto sem processamento
if not os.path.exists(MASK):
    print("  Mask nao encontrada, copiando video sem watermark removal...")
    shutil.copy2(INPUT, OUTPUT)
    count = 0
else:
    cap = cv2.VideoCapture(INPUT)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    FPS = cap.get(cv2.CAP_PROP_FPS)
    TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    mask_np = cv2.imread(MASK, cv2.IMREAD_GRAYSCALE)
    mask_np = cv2.resize(mask_np, (W, H))
    _, mask_bin = cv2.threshold(mask_np, 10, 255, cv2.THRESH_BINARY)

    pipe = subprocess.Popen([
        "ffmpeg", "-y", "-f", "rawvideo", "-vcodec", "rawvideo",
        "-s", f"{W}x{H}", "-pix_fmt", "bgr24", "-r", str(FPS), "-i", "pipe:0",
        "-c:v", "h264_nvenc", "-preset", "p2", "-b:v", "5M", "-c:a", "copy", OUTPUT
    ], stdin=subprocess.PIPE)

    count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        out = cv2.inpaint(frame, mask_bin, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
        pipe.stdin.write(out.tobytes())
        count += 1
        if count % 500 == 0:
            print(f"  Frame {count}/{TOTAL} ({count/TOTAL*100:.1f}%)")

    cap.release()
    pipe.stdin.close()
    pipe.wait()
print(f"  {count} frames processados")

cell_end(2, 'done', 'Processamento Watermark concluido')

In [ ]:
cell_start(3, 'Upload Resultado')

salvar_no_drive(f"{BASE_PATH}/pt5_limpo.mp4", f"{DRIVE_WATERMARK}/pt5_limpo.mp4")

cell_end(3, 'done', 'Upload Resultado concluido')

In [ ]:
cell_start(4, 'Finalizacao')

report_step("done", f"Watermark PT5 concluido - {count} frames")

cell_end(4, 'done', 'Finalizacao concluido')